# LeetCode 1405 — Longest Happy String bằng Python

Link đề: https://leetcode.com/problems/longest-happy-string/ · Độ khó: **Medium** · Chủ đề: String, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def longestDiverseString(self, a: int, b: int, c: int) -> str:
        ...
```

> Bài có **nhiều đáp án đúng**. Notebook kiểm tra bằng hàm `is_happy` (tính hợp lệ) và so **độ dài** với lời giải chuẩn (brute force / DP / công thức), không so chuỗi cố định.

## 1. Mục tiêu học

1. Hiểu tham lam "**ưu tiên ký tự còn nhiều nhất, nếu bị chặn thì lấy ký tự kế tiếp**" và vì sao nó tối ưu.
2. Dùng `heapq` làm **max-heap** (lưu số âm) và kỹ thuật "lấy ra – dùng – trả lại" phần tử.
3. Đi từ brute force (quay lui mọi chuỗi) → DP có nhớ → tham lam heap → tham lam không heap → dựng đệ quy.
4. Tìm và chứng minh **công thức độ dài tối đa**: với `L` = số lớn nhất, `R` = tổng hai số còn lại, độ dài = `min(L + R, 2(R + 1) + R)`.
5. Kiểm thử bài nhiều đáp án: viết **validator** + so độ dài; tránh các lỗi "luôn lấy 2 ký tự", quên kiểm tra 2 ký tự cuối, quên dừng khi bị kẹt.

## 2. Đề bài & ví dụ

Chuỗi `s` gọi là **happy** nếu:

- chỉ gồm các chữ `'a'`, `'b'`, `'c'`;
- **không** chứa chuỗi con `"aaa"`, `"bbb"`, `"ccc"` (không có 3 chữ giống nhau liền nhau);
- có **tối đa** `a` chữ `'a'`, `b` chữ `'b'`, `c` chữ `'c'`.

Cho `a, b, c`, trả về chuỗi happy **dài nhất** (nhiều đáp án thì trả bất kỳ); nếu không có thì trả `""`.

**Ví dụ 1:** `a = 1, b = 1, c = 7` → `"ccaccbcc"` (cũng đúng: `"ccbccacc"`).

**Ví dụ 2:** `a = 7, b = 1, c = 0` → `"aabaa"` (đáp án duy nhất).

**Ràng buộc:** `0 <= a, b, c <= 100`, `a + b + c > 0`.

In [1]:
import heapq
import random
import time
from functools import lru_cache
from typing import List

examples = [((1, 1, 7), 8), ((7, 1, 0), 5)]     # (a, b, c), độ dài tối đa

def is_happy(s, a, b, c):
    """Kiểm tra s có phải chuỗi happy với giới hạn a, b, c."""
    if any(ch not in "abc" for ch in s):
        return False
    if s.count("a") > a or s.count("b") > b or s.count("c") > c:
        return False
    return all(not (s[i] == s[i + 1] == s[i + 2]) for i in range(len(s) - 2))

for t in ["ccaccbcc", "ccbccacc", "aabaa", "aaab", "ccacccbc"]:
    print(t, "happy với (1,1,7)?", is_happy(t, 1, 1, 7), "| với (7,1,0)?", is_happy(t, 7, 1, 0))

ccaccbcc happy với (1,1,7)? True | với (7,1,0)? False
ccbccacc happy với (1,1,7)? True | với (7,1,0)? False
aabaa happy với (1,1,7)? False | với (7,1,0)? True
aaab happy với (1,1,7)? False | với (7,1,0)? False
ccacccbc happy với (1,1,7)? False | với (7,1,0)? False


## 3. Kiến thức nền

### 3.1 `heapq` là min-heap — muốn max-heap thì lưu số âm

`heapq.heappush(h, x)`, `heapq.heappop(h)` lấy phần tử **nhỏ nhất**. Lưu `(-count, ch)` thì phần tử "nhỏ nhất" là ký tự có `count` **lớn nhất**.

In [2]:
h = []
for cnt, ch in [(1, "a"), (1, "b"), (7, "c")]:
    heapq.heappush(h, (-cnt, ch))
print("heap:", h)
neg, ch = heapq.heappop(h)
print("lấy ra:", ch, "còn", -neg, "| heap còn:", h)

heap: [(-7, 'c'), (-1, 'b'), (-1, 'a')]
lấy ra: c còn 7 | heap còn: [(-1, 'a'), (-1, 'b')]


### 3.2 Kiểm tra 2 ký tự cuối

`s[-2:] == ch * 2` (hoặc `len(s) >= 2 and s[-1] == s[-2] == ch`). Với list ký tự `res`, dùng `res[-2:] == [ch, ch]`. Slicing an toàn cả khi chuỗi ngắn hơn 2.

In [3]:
for s in ["", "c", "cc", "cca", "acc"]:
    print(repr(s), "kết thúc bằng 'cc'?", s[-2:] == "cc")

'' kết thúc bằng 'cc'? False
'c' kết thúc bằng 'cc'? False
'cc' kết thúc bằng 'cc'? True
'cca' kết thúc bằng 'cc'? False
'acc' kết thúc bằng 'cc'? True


### 3.3 Ghép chuỗi bằng list rồi `"".join`

Nối `s += ch` lặp nhiều lần có thể tốn `O(n²)`; gom vào list rồi `"".join(res)` là `O(n)`.

In [4]:
res = []
for ch in "ccaccbcc":
    res.append(ch)
print("".join(res), res[-2:])

ccaccbcc ['c', 'c']


### 3.4 `lru_cache` cho đệ quy có nhớ

`@lru_cache(maxsize=None)` ghi nhớ kết quả theo tham số; hàm phải có tham số **hashable** (số, chuỗi, tuple).

In [5]:
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(80), fib.cache_info().hits)

23416728348467685 78


### 3.5 Sắp xếp theo khoá giảm dần

Với chỉ 3 ký tự, mỗi bước có thể sắp xếp lại `sorted(items, key=lambda x: -x[0])` — `O(3 log 3)` = hằng số, không cần heap.

In [6]:
items = [[1, "a"], [1, "b"], [7, "c"]]
print(sorted(items, key=lambda x: -x[0]))

[[7, 'c'], [1, 'a'], [1, 'b']]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 2** `a = 7, b = 1, c = 0`: chữ `a` quá nhiều. Mỗi khối `aa` phải được ngăn cách bằng một chữ khác. Chỉ có 1 chữ `b` ⇒ tối đa 2 khối `aa` ⇒ `aa b aa` = 5 ký tự, bỏ thừa 3 chữ `a`.

Tổng quát: gọi `L` = số lượng chữ nhiều nhất, `R` = tổng hai chữ còn lại. Các chữ "khác" tạo ra tối đa `R + 1` khe (trước, giữa, sau), mỗi khe chứa ≤ 2 chữ lớn ⇒ dùng được tối đa `2(R + 1)` chữ lớn. Vậy:

- nếu `L ≤ 2(R + 1)` ⇒ có thể dùng **hết** `L + R` ký tự;
- ngược lại ⇒ độ dài `2(R + 1) + R`.

**Ví dụ 1** `(1, 1, 7)`: `L = 7`, `R = 2`, `2(R + 1) = 6 < 7` ⇒ độ dài `6 + 2 = 8` ✔ (`"ccaccbcc"` dùng 6 chữ `c`).

Chiến lược tự nhiên: **luôn đặt chữ còn nhiều nhất**, trừ khi đặt nữa sẽ thành 3 chữ liền ⇒ đặt chữ nhiều thứ hai (đóng vai "vách ngăn").

In [7]:
def max_len_formula(a, b, c):
    L = max(a, b, c)
    R = a + b + c - L
    return min(L + R, 2 * (R + 1) + R)

for (a, b, c), want in examples + [((0, 8, 11), None), ((2, 2, 1), None), ((0, 0, 5), None)]:
    print((a, b, c), "-> độ dài tối đa", max_len_formula(a, b, c), "" if want is None else f"(đề: {want})")

(1, 1, 7) -> độ dài tối đa 8 (đề: 8)
(7, 1, 0) -> độ dài tối đa 5 (đề: 5)
(0, 8, 11) -> độ dài tối đa 19 
(2, 2, 1) -> độ dài tối đa 5 
(0, 0, 5) -> độ dài tối đa 2 


## 5. Cách 1 — Brute Force / baseline: quay lui mọi chuỗi happy

### Ý tưởng
Xây chuỗi từng ký tự: ở mỗi bước thử thêm `'a'`, `'b'`, `'c'` nếu còn và không tạo 3 chữ liền. Ghi nhận chuỗi dài nhất từng gặp. Đây là duyệt **toàn bộ** cây chuỗi happy — số nút tăng theo hàm mũ.

In [8]:
def happy_brute(a, b, c):
    cnt = {"a": a, "b": b, "c": c}
    best = [""]
    cur = []

    def dfs():
        if len(cur) > len(best[0]):
            best[0] = "".join(cur)
        for ch in "abc":
            if cnt[ch] == 0 or cur[-2:] == [ch, ch]:
                continue
            cnt[ch] -= 1; cur.append(ch)
            dfs()
            cur.pop(); cnt[ch] += 1

    dfs()
    return best[0]

for (a, b, c), want in examples:
    r = happy_brute(a, b, c)
    print((a, b, c), "->", r, "| độ dài", len(r), "(mong đợi", want, ") | happy?", is_happy(r, a, b, c))

(1, 1, 7) -> ccaccbcc | độ dài 8 (mong đợi 8 ) | happy? True
(7, 1, 0) -> aabaa | độ dài 5 (mong đợi 5 ) | happy? True


### Độ phức tạp và nút thắt

Số chuỗi happy có thể lên tới cỡ `3^(a+b+c)` ⇒ Time **hàm mũ**. Đo thử:

In [9]:
for abc in [(2, 2, 2), (3, 3, 3), (3, 4, 4)]:
    t = time.perf_counter()
    r = happy_brute(*abc)
    print(abc, "độ dài", len(r), f"({time.perf_counter() - t:.3f}s)")

(2, 2, 2) độ dài 6 (0.000s)
(3, 3, 3) độ dài 9 (0.002s)
(3, 4, 4) độ dài 11 (0.014s)


Nút thắt: rất nhiều nhánh có **cùng trạng thái** (còn bao nhiêu mỗi chữ, chữ cuối là gì, đã lặp mấy lần) nhưng bị duyệt lại. ⇒ DP có nhớ (Cách 2) hoặc tìm quy tắc chọn tham lam (Cách 3).

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái đủ dùng:** `(còn_a, còn_b, còn_c, chữ_cuối, số_lần_lặp_cuối)` — tương lai chỉ phụ thuộc chừng đó ⇒ DP.

**Insight tham lam:** chữ nhiều nhất là "mối nguy" — nếu để dành nó, cuối cùng sẽ thừa nhiều mà không còn vách ngăn. Vậy:

- **Nếu** chữ nhiều nhất đặt được (2 ký tự cuối không phải chính nó) **thì** đặt **một** chữ đó.
- **Nếu** nó bị chặn **thì** đặt **một** chữ nhiều thứ hai (nếu còn); nếu không còn chữ nào khác ⇒ **dừng**.

**Bất biến:** sau mỗi bước, chuỗi đang xây là happy; và (chứng minh ở mục 10) độ dài cuối đạt đúng công thức `min(L + R, 3R + 2)`.

Chú ý: đặt **một** chữ mỗi lần (rồi xét lại) an toàn hơn "đặt luôn 2 chữ" — đặt 2 chữ một lúc có thể lãng phí vách ngăn (xem Lỗi 1).

## 7. Các cách tiếp theo

### Cách 2 — DP có nhớ trên trạng thái

`best(a, b, c, last, run)` = chuỗi happy dài nhất có thể **nối thêm** khi còn `a, b, c` và chuỗi hiện tại kết thúc bằng `run` chữ `last`. Số trạng thái `≈ 101³ · 4 · 3` — quá nhiều cho Python với giới hạn 100, nhưng **chính xác tuyệt đối** và nhanh với số nhỏ ⇒ dùng để kiểm chứng tham lam.

In [10]:
def happy_dp(a, b, c):
    @lru_cache(maxsize=None)
    def best(a, b, c, last, run):
        res = ""
        for i, ch in enumerate("abc"):
            left = (a, b, c)[i]
            if left == 0 or (ch == last and run == 2):
                continue
            na, nb, nc = (a - (i == 0), b - (i == 1), c - (i == 2))
            cand = ch + best(na, nb, nc, ch, run + 1 if ch == last else 1)
            if len(cand) > len(res):
                res = cand
        return res
    return best(a, b, c, "", 0)

for (a, b, c), want in examples:
    r = happy_dp(a, b, c)
    print((a, b, c), "->", r, len(r), "| happy?", is_happy(r, a, b, c))
t = time.perf_counter(); r = happy_dp(12, 12, 12); print("(12,12,12) độ dài", len(r), f"({time.perf_counter() - t:.3f}s)")

(1, 1, 7) -> ccaccbcc 8 | happy? True
(7, 1, 0) -> aabaa 5 | happy? True
(12,12,12) độ dài 36 (0.017s)


Time `O(a·b·c)` trạng thái × `O(len)` để nối chuỗi; Space tương ứng. Chỉ dùng cho số nhỏ.

### Cách 3 — Tham lam với max-heap

**Từng đoạn:**
1. Đưa `(-count, ch)` của các chữ có `count > 0` vào heap.
2. Lặp: lấy chữ nhiều nhất `x`.
   - Nếu 2 ký tự cuối là `x x`: cần chữ khác. Nếu heap rỗng ⇒ dừng (trả `x` lại cũng được, nhưng không cần). Ngược lại lấy chữ kế `y`, đặt `y`, giảm đếm, trả `y` (nếu còn) và trả `x` về heap.
   - Ngược lại: đặt `x`, giảm đếm, trả `x` về heap nếu còn.
3. Ghép kết quả.

In [11]:
def happy_heap(a, b, c):
    h = [(-cnt, ch) for cnt, ch in ((a, "a"), (b, "b"), (c, "c")) if cnt > 0]
    heapq.heapify(h)
    res = []
    while h:
        n1, c1 = heapq.heappop(h)
        if res[-2:] == [c1, c1]:                  # chữ nhiều nhất bị chặn
            if not h:
                break                             # không còn vách ngăn -> dừng
            n2, c2 = heapq.heappop(h)
            res.append(c2)
            if n2 + 1 < 0:
                heapq.heappush(h, (n2 + 1, c2))
            heapq.heappush(h, (n1, c1))           # trả chữ nhiều nhất về
        else:
            res.append(c1)
            if n1 + 1 < 0:
                heapq.heappush(h, (n1 + 1, c1))
    return "".join(res)

for (a, b, c), want in examples:
    r = happy_heap(a, b, c)
    print((a, b, c), "->", r, len(r), "| happy?", is_happy(r, a, b, c), "| đủ dài?", len(r) == want)

(1, 1, 7) -> ccaccbcc 8 | happy? True | đủ dài? True
(7, 1, 0) -> aabaa 5 | happy? True | đủ dài? True


In [12]:
class SolutionHeap:
    def longestDiverseString(self, a: int, b: int, c: int) -> str:
        h = [(-cnt, ch) for cnt, ch in ((a, "a"), (b, "b"), (c, "c")) if cnt > 0]
        heapq.heapify(h)
        res = []
        while h:
            n1, c1 = heapq.heappop(h)
            if len(res) >= 2 and res[-1] == res[-2] == c1:
                if not h:
                    break
                n2, c2 = heapq.heappop(h)
                res.append(c2)
                if n2 + 1:
                    heapq.heappush(h, (n2 + 1, c2))
                heapq.heappush(h, (n1, c1))
            else:
                res.append(c1)
                if n1 + 1:
                    heapq.heappush(h, (n1 + 1, c1))
        return "".join(res)

print([SolutionHeap().longestDiverseString(*abc) for abc, _ in examples])

['ccaccbcc', 'aabaa']


Time `O(N log 3)` = `O(N)` với `N = a + b + c` (heap tối đa 3 phần tử), Space `O(N)` cho kết quả.

### Cách 4 — Tham lam không cần heap (sắp xếp 3 phần tử mỗi bước)

Chỉ có 3 chữ ⇒ mỗi bước sắp xếp lại danh sách `[count, ch]` giảm dần, rồi duyệt theo thứ tự: chữ **đầu tiên** còn hàng và không bị chặn thì đặt. Không tìm được ⇒ dừng. Ngắn gọn, dễ nhớ.

In [13]:
def happy_sort(a, b, c):
    items = [[a, "a"], [b, "b"], [c, "c"]]
    res = []
    while True:
        items.sort(key=lambda x: -x[0])
        for it in items:
            cnt, ch = it
            if cnt == 0:
                continue
            if res[-2:] == [ch, ch]:
                continue
            res.append(ch)
            it[0] -= 1
            break
        else:
            break                                   # không chữ nào đặt được
    return "".join(res)

for (a, b, c), want in examples:
    r = happy_sort(a, b, c)
    print((a, b, c), "->", r, len(r), "| happy?", is_happy(r, a, b, c))

(1, 1, 7) -> ccaccbcc 8 | happy? True
(7, 1, 0) -> aabaa 5 | happy? True


Lưu ý cấu trúc `for … else`: khối `else` chạy khi vòng `for` **không** gặp `break` — tức là không chữ nào đặt được ⇒ dừng vòng `while`. Time `O(N)`, Space `O(N)`.

### Cách 5 — Dựng đệ quy theo khối

Luôn gọi chữ nhiều nhất là `A` (đổi chỗ tham số cho tới khi `a ≥ b ≥ c`):
- Nếu `b == 0`: chỉ còn `A` ⇒ trả `A * min(2, a)`.
- Ngược lại đặt `min(2, a)` chữ `A`; nếu sau đó `A` vẫn **không ít hơn** `B` thì đặt thêm **một** chữ `B` làm vách ngăn; gọi đệ quy với số lượng còn lại.

Mỗi lần gọi tiêu thụ ít nhất 1 chữ ⇒ độ sâu đệ quy ≤ `a + b + c ≤ 300` (an toàn).

In [14]:
def happy_rec(a, b, c, A="a", B="b", C="c"):
    if a < b:
        return happy_rec(b, a, c, B, A, C)
    if b < c:
        return happy_rec(a, c, b, A, C, B)
    if b == 0:
        return A * min(2, a)
    use_a = min(2, a)
    use_b = 1 if a - use_a >= b else 0
    return A * use_a + B * use_b + happy_rec(a - use_a, b - use_b, c, A, B, C)

for (a, b, c), want in examples:
    r = happy_rec(a, b, c)
    print((a, b, c), "->", r, len(r), "| happy?", is_happy(r, a, b, c))

(1, 1, 7) -> ccaccbcc 8 | happy? True
(7, 1, 0) -> aabaa 5 | happy? True


Vì sao chỗ nối giữa hai lần gọi không tạo 3 chữ liền? Nếu đã đặt vách `B` thì lần gọi sau bắt đầu bằng chữ nhiều nhất — có thể là `A` nhưng đã bị `B` ngăn. Nếu **không** đặt `B` (vì `a − 2 < b`), thì ở lần gọi sau `B` nhiều hơn `A` hẳn ⇒ chuỗi tiếp theo bắt đầu bằng `B`, không phải `A`. Time `O(N)`, Space `O(N)` (đệ quy + chuỗi).

## 8. TRACE từng bước

Trace Cách 3 (heap) trên **Ví dụ 1** `(a, b, c) = (1, 1, 7)`:

| Bước | Heap trước (đếm, chữ) | Lấy ra | 2 ký tự cuối | Đặt | Kết quả |
|---|---|---|---|---|---|
| 1 | c7 a1 b1 | c | — | c | c |
| 2 | c6 a1 b1 | c | `c` | c | cc |
| 3 | c5 a1 b1 | c (bị chặn) → a | `cc` | a | cca |
| 4 | c5 b1 | c | `ca` | c | ccac |
| 5 | c4 b1 | c | `ac` | c | ccacc |
| 6 | c3 b1 | c (bị chặn) → b | `cc` | b | ccaccb |
| 7 | c3 | c | `cb` | c | ccaccbc |
| 8 | c2 | c | `bc` | c | ccaccbcc |
| 9 | c1 | c (bị chặn), heap rỗng | `cc` | dừng | **ccaccbcc** |

In [15]:
def trace_heap(a, b, c):
    h = [(-cnt, ch) for cnt, ch in ((a, "a"), (b, "b"), (c, "c")) if cnt > 0]
    heapq.heapify(h)
    res, step = [], 0
    while h:
        step += 1
        state = " ".join(f"{ch}{-n}" for n, ch in sorted(h))
        n1, c1 = heapq.heappop(h)
        if res[-2:] == [c1, c1]:
            if not h:
                print(f"bước {step}: heap [{state}] lấy {c1} nhưng bị chặn, heap rỗng -> DỪNG")
                break
            n2, c2 = heapq.heappop(h)
            res.append(c2)
            if n2 + 1 < 0:
                heapq.heappush(h, (n2 + 1, c2))
            heapq.heappush(h, (n1, c1))
            print(f"bước {step}: heap [{state}] lấy {c1} bị chặn -> đặt {c2} | {''.join(res)}")
        else:
            res.append(c1)
            if n1 + 1 < 0:
                heapq.heappush(h, (n1 + 1, c1))
            print(f"bước {step}: heap [{state}] đặt {c1} | {''.join(res)}")
    return "".join(res)

r = trace_heap(1, 1, 7)
print("kết quả:", r, "| độ dài", len(r), "= công thức", max_len_formula(1, 1, 7))

bước 1: heap [c7 a1 b1] đặt c | c
bước 2: heap [c6 a1 b1] đặt c | cc
bước 3: heap [c5 a1 b1] lấy c bị chặn -> đặt a | cca
bước 4: heap [c5 b1] đặt c | ccac
bước 5: heap [c4 b1] đặt c | ccacc
bước 6: heap [c3 b1] lấy c bị chặn -> đặt b | ccaccb
bước 7: heap [c3] đặt c | ccaccbc
bước 8: heap [c2] đặt c | ccaccbcc
bước 9: heap [c1] lấy c nhưng bị chặn, heap rỗng -> DỪNG
kết quả: ccaccbcc | độ dài 8 = công thức 8


Trace thêm một trường hợp **dùng hết** mọi ký tự `(a, b, c) = (2, 2, 1)` và trường hợp chỉ có một chữ `(0, 0, 5)`:

In [16]:
r = trace_heap(2, 2, 1); print("->", r, "dùng hết?", len(r) == 5)
print()
r = trace_heap(0, 0, 5); print("->", repr(r))

bước 1: heap [a2 b2 c1] đặt a | a
bước 2: heap [b2 a1 c1] đặt b | ab
bước 3: heap [a1 b1 c1] đặt a | aba
bước 4: heap [b1 c1] đặt b | abab
bước 5: heap [c1] đặt c | ababc
-> ababc dùng hết? True

bước 1: heap [c5] đặt c | c
bước 2: heap [c4] đặt c | cc
bước 3: heap [c3] lấy c nhưng bị chặn, heap rỗng -> DỪNG
-> 'cc'


## 9. Code nộp LeetCode

Dùng Cách 3 (tham lam với heap).

In [17]:
import heapq

class Solution:
    def longestDiverseString(self, a: int, b: int, c: int) -> str:
        heap = [(-cnt, ch) for cnt, ch in ((a, "a"), (b, "b"), (c, "c")) if cnt > 0]
        heapq.heapify(heap)                       # max-heap theo số lượng còn lại
        res = []
        while heap:
            n1, c1 = heapq.heappop(heap)
            if len(res) >= 2 and res[-1] == res[-2] == c1:
                if not heap:                      # bị chặn và không còn chữ khác
                    break
                n2, c2 = heapq.heappop(heap)      # dùng chữ nhiều thứ hai làm vách ngăn
                res.append(c2)
                if n2 + 1 < 0:
                    heapq.heappush(heap, (n2 + 1, c2))
                heapq.heappush(heap, (n1, c1))
            else:
                res.append(c1)
                if n1 + 1 < 0:
                    heapq.heappush(heap, (n1 + 1, c1))
        return "".join(res)

for (a, b, c), want in examples:
    r = Solution().longestDiverseString(a, b, c)
    assert is_happy(r, a, b, c) and len(r) == want
    print((a, b, c), "->", r)
print("Solution đúng với 2 ví dụ của đề (kiểm tra hợp lệ + độ dài)")

(1, 1, 7) -> ccaccbcc
(7, 1, 0) -> aabaa
Solution đúng với 2 ví dụ của đề (kiểm tra hợp lệ + độ dài)


## 10. Vì sao thuật toán đúng

**Cận trên (không thể dài hơn).** Gọi chữ nhiều nhất có `L` chữ, hai chữ kia tổng `R`. Trong một chuỗi happy, giữa hai chữ "khác" liên tiếp (và ở hai đầu) có tối đa 2 chữ lớn. Có tối đa `R` chữ khác ⇒ tối đa `R + 1` khe ⇒ tối đa `2(R + 1)` chữ lớn. Độ dài ≤ `min(L, 2(R + 1)) + R = min(L + R, 3R + 2)`.

**Tham lam đạt cận trên.** Xét thuật toán đặt từng chữ.
- *Hợp lệ:* chỉ đặt `x` khi 2 ký tự cuối không phải `xx` ⇒ không bao giờ có 3 chữ liền; không đặt chữ đã hết ⇒ không vượt giới hạn.
- *Không dừng sớm vô lý:* thuật toán chỉ dừng khi chữ nhiều nhất `x` bị chặn **và** hai chữ còn lại đã hết. Khi đó phần dư chỉ còn chữ `x`.
- *Phần dư đó là tối thiểu:* Nếu lúc đầu `L ≤ 2(R + 1)`, luôn ưu tiên chữ nhiều nhất giữ được tính "cân bằng": bất biến "chữ nhiều nhất còn lại ≤ 2 × (tổng hai chữ kia còn lại + 1)" được bảo toàn (đặt chữ lớn làm vế trái giảm; khi bị chặn và đặt vách, chữ lớn vừa được đặt 2 lần liền, vế phải giảm 1 nhưng vế trái đã giảm 2 trước đó). Do đó không bao giờ rơi vào cảnh còn ≥ 3 chữ `x` mà hết vách ⇒ dùng hết `L + R`. Nếu `L > 2(R + 1)`, chữ lớn luôn là nhiều nhất từ đầu đến cuối (nó giảm chậm hơn tổng hai chữ kia không đáng kể), thuật toán tạo đúng mẫu `xx y xx y … xx`, dùng hết `R` vách và `2(R + 1)` chữ lớn — bằng cận trên.

Phần chứng minh "bất biến cân bằng" ở trên là phác thảo; mục 12 kiểm chứng **vét cạn** mọi bộ `(a, b, c)` nhỏ với DP chính xác, và mọi bộ tới 100 với công thức.

## 11. Phân tích Time / Space

Gọi `N = a + b + c` (≤ 300).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Quay lui | hàm mũ (~`3^N`) | `O(N)` | chỉ số rất nhỏ |
| 2. DP có nhớ | `O(a·b·c·12·N)` (nối chuỗi) | `O(a·b·c·N)` | chính xác, số nhỏ |
| 3. Tham lam heap | `O(N log 3)` = `O(N)` | `O(N)` | chuẩn |
| 4. Tham lam sắp xếp | `O(N · 3 log 3)` = `O(N)` | `O(N)` | không cần heap |
| 5. Dựng đệ quy | `O(N)` lời gọi (mỗi lời gọi nối chuỗi) | `O(N)` | ngắn, cần lập luận nối khối |

## 12. Bộ test

Vì có nhiều đáp án, mọi kiểm tra đều dạng: **(1)** kết quả là chuỗi happy, **(2)** độ dài bằng độ dài tối ưu.

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {
    "heap": happy_heap,
    "heap_submit": lambda a, b, c: SolutionHeap().longestDiverseString(a, b, c),
    "sort": happy_sort,
    "rec": happy_rec,
    "submit": lambda a, b, c: Solution().longestDiverseString(a, b, c),
}
edge = [
    ((1, 0, 0), 1), ((0, 0, 2), 2), ((0, 0, 3), 2), ((0, 0, 100), 2),
    ((1, 1, 1), 3), ((2, 2, 2), 6),
    ((0, 1, 5), 5),          # cc b cc
    ((0, 2, 5), 7),          # dùng hết
    ((100, 100, 100), 300),
    ((100, 0, 0), 2), ((100, 1, 0), 5), ((100, 50, 0), 150),
    ((100, 10, 10), 62),     # 2*(20+1) + 20
]
def check(f, a, b, c, want):
    r = f(a, b, c)
    return is_happy(r, a, b, c) and len(r) == want

for (a, b, c), want in edge:
    assert max_len_formula(a, b, c) == want, (a, b, c)
for name, f in ALL.items():
    for (a, b, c), want in examples + edge:
        assert check(f, a, b, c, want), (name, a, b, c, f(a, b, c))
print("Mọi cách đúng với", len(examples + edge), "test (hợp lệ + đúng độ dài)")

Mọi cách đúng với 15 test (hợp lệ + đúng độ dài)


### 12.2 Vét cạn mọi bộ nhỏ với DP chính xác (và brute force cho bộ rất nhỏ)

In [19]:
cnt = 0
for a in range(8):
    for b in range(8):
        for c in range(8):
            if a + b + c == 0:
                continue
            opt = len(happy_dp(a, b, c))
            assert opt == max_len_formula(a, b, c), (a, b, c)
            if a + b + c <= 7:
                assert len(happy_brute(a, b, c)) == opt
            for name, f in ALL.items():
                assert check(f, a, b, c, opt), (name, a, b, c)
            cnt += 1
print("vét cạn", cnt, "bộ (a,b,c) <= 7: DP = công thức = brute force; mọi cách tham lam đạt tối ưu")

vét cạn 511 bộ (a,b,c) <= 7: DP = công thức = brute force; mọi cách tham lam đạt tối ưu


### 12.3 Toàn bộ miền ràng buộc với công thức (seed cố định cho mẫu ngẫu nhiên)

In [20]:
rng = random.Random(1405)
t = time.perf_counter()
for _ in range(3000):
    a, b, c = rng.randint(0, 100), rng.randint(0, 100), rng.randint(0, 100)
    if a + b + c == 0:
        continue
    want = max_len_formula(a, b, c)
    for name, f in ALL.items():
        assert check(f, a, b, c, want), (name, a, b, c)
# các bộ "lệch" nhiều nhất
for L in range(0, 101, 5):
    for R1 in range(0, 6):
        for R2 in range(0, 6):
            if L + R1 + R2:
                for perm in [(L, R1, R2), (R1, L, R2), (R1, R2, L)]:
                    assert check(Solution().longestDiverseString, *perm, max_len_formula(*perm))
print(f"3000 bộ ngẫu nhiên + các bộ lệch: đều hợp lệ và tối ưu ({time.perf_counter() - t:.2f}s)")

3000 bộ ngẫu nhiên + các bộ lệch: đều hợp lệ và tối ưu (0.80s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Mỗi lượt đặt luôn **2** chữ của chữ nhiều nhất

Ý tưởng "đặt càng nhiều càng tốt" nghe hợp lý, nhưng có thể dùng vách ngăn quá sớm và lãng phí.

In [21]:
def wrong_take_two(a, b, c):
    cnt = {"a": a, "b": b, "c": c}
    s = ""
    while True:
        order = sorted((ch for ch in cnt if cnt[ch] > 0), key=lambda ch: -cnt[ch])
        for ch in order:
            if s.endswith(ch):
                continue
            k = min(2, cnt[ch])                       # SAI: luôn lấy 2
            s += ch * k; cnt[ch] -= k
            break
        else:
            break
    return s

for abc in [(0, 2, 5), (5, 2, 0), (2, 0, 5)]:
    w = wrong_take_two(*abc)
    r = Solution().longestDiverseString(*abc)
    print(abc, "| sai:", w, len(w), "| đúng:", r, len(r))

(0, 2, 5) | sai: ccbbcc 6 | đúng: ccbccbc 7
(5, 2, 0) | sai: aabbaa 6 | đúng: aabaaba 7
(2, 0, 5) | sai: ccaacc 6 | đúng: ccaccac 7


`(0, 2, 5)`: bản sai `cc bb cc` (6 ký tự) — đặt luôn hai chữ `b` liền nhau nên chỉ còn một "vách" ⇒ thừa 1 chữ `c`. Đáp án tối ưu (ví dụ `ccbccbc`) dài 7. **Sửa:** mỗi lượt đặt **một** chữ rồi đánh giá lại.

### Lỗi 2 — Quên kiểm tra 2 ký tự cuối

In [22]:
def wrong_no_check(a, b, c):
    h = [(-n, ch) for n, ch in ((a, "a"), (b, "b"), (c, "c")) if n > 0]
    heapq.heapify(h)
    res = []
    while h:
        n1, c1 = heapq.heappop(h)
        res.append(c1)                                # SAI: không kiểm tra "xx"
        if n1 + 1 < 0:
            heapq.heappush(h, (n1 + 1, c1))
    return "".join(res)

for abc in [(1, 1, 7), (7, 1, 0)]:
    w = wrong_no_check(*abc)
    print(abc, "| sai:", w, "happy?", is_happy(w, *abc), "| đúng:", Solution().longestDiverseString(*abc))

(1, 1, 7) | sai: ccccccabc happy? False | đúng: ccaccbcc
(7, 1, 0) | sai: aaaaaaab happy? False | đúng: aabaa


Chuỗi trả về chứa `"ccc"` / `"aaa"` ⇒ không hợp lệ. **Sửa:** chặn khi `res[-2:] == [c1, c1]`.

### Lỗi 3 — Bị chặn nhưng không dừng (lặp vô hạn)

Khi chữ nhiều nhất bị chặn và heap rỗng, nếu chỉ "trả lại" chữ đó rồi `continue`, vòng lặp chạy mãi. Bản minh hoạ có bộ đếm để tự dừng.

In [23]:
def wrong_never_stop(a, b, c, limit=1000):
    h = [(-n, ch) for n, ch in ((a, "a"), (b, "b"), (c, "c")) if n > 0]
    heapq.heapify(h)
    res, it = [], 0
    while h:
        it += 1
        if it > limit:
            return f"lặp {limit} vòng chưa xong (vô hạn); res = {''.join(res)}"
        n1, c1 = heapq.heappop(h)
        if res[-2:] == [c1, c1]:
            if h:
                n2, c2 = heapq.heappop(h)
                res.append(c2)
                if n2 + 1 < 0:
                    heapq.heappush(h, (n2 + 1, c2))
            heapq.heappush(h, (n1, c1))               # SAI: heap rỗng vẫn trả lại rồi lặp tiếp
            continue
        res.append(c1)
        if n1 + 1 < 0:
            heapq.heappush(h, (n1 + 1, c1))
    return "".join(res)

print("sai :", wrong_never_stop(7, 1, 0))
print("đúng:", Solution().longestDiverseString(7, 1, 0))

sai : lặp 1000 vòng chưa xong (vô hạn); res = aabaa
đúng: aabaa


**Sửa:** nếu bị chặn và không còn chữ khác ⇒ `break`.

### Lỗi 4 — Quên rằng `heapq` là **min-heap**

Lưu `(count, ch)` (không đổi dấu) ⇒ luôn lấy chữ **ít nhất** trước ⇒ tiêu hết vách ngăn sớm.

In [24]:
def wrong_min_heap(a, b, c):
    h = [(n, ch) for n, ch in ((a, "a"), (b, "b"), (c, "c")) if n > 0]   # SAI: không đổi dấu
    heapq.heapify(h)
    res = []
    while h:
        n1, c1 = heapq.heappop(h)
        if res[-2:] == [c1, c1]:
            if not h:
                break
            n2, c2 = heapq.heappop(h)
            res.append(c2)
            if n2 - 1 > 0:
                heapq.heappush(h, (n2 - 1, c2))
            heapq.heappush(h, (n1, c1))
        else:
            res.append(c1)
            if n1 - 1 > 0:
                heapq.heappush(h, (n1 - 1, c1))
    return "".join(res)

for abc in [(1, 1, 7), (7, 1, 0), (0, 2, 5)]:
    w = wrong_min_heap(*abc)
    print(abc, "| sai:", w, len(w), "| tối ưu:", max_len_formula(*abc))

(1, 1, 7) | sai: abcc 4 | tối ưu: 8
(7, 1, 0) | sai: baa 3 | tối ưu: 5
(0, 2, 5) | sai: bbcc 4 | tối ưu: 7


**Sửa:** lưu `(-count, ch)`.

### Lỗi 5 — Tự test bằng cách so với **một** chuỗi cố định

Đề có nhiều đáp án. Nếu chép chuỗi mẫu `"ccbccacc"` (đề cũng nêu là đúng) rồi so `==`, ta sẽ đánh trượt lời giải `"ccaccbcc"` hoàn toàn đúng.

In [25]:
r = happy_rec(1, 1, 7)
print("kết quả Cách 5:", r)
print("so với chuỗi mẫu 'ccbccacc' (cũng là đáp án đề nêu):", r == "ccbccacc")
print("so đúng cách    :", is_happy(r, 1, 1, 7) and len(r) == max_len_formula(1, 1, 7))

kết quả Cách 5: ccaccbcc
so với chuỗi mẫu 'ccbccacc' (cũng là đáp án đề nêu): False
so đúng cách    : True


**Sửa:** viết validator `is_happy` + so độ dài với công thức/DP.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Xếp các phần tử sao cho không có k cái giống nhau liền nhau, dài nhất/được không?"** ⇒ tham lam: ưu tiên loại **còn nhiều nhất**, bị chặn thì lấy loại kế.
- Đếm theo loại + chọn max liên tục ⇒ **heap** (`(-count, item)`); ít loại thì sắp xếp thẳng.
- Đặt **từng phần tử một** rồi đánh giá lại an toàn hơn đặt theo khối.
- Luôn có công thức kiểm tra: loại lớn nhất `L`, phần còn lại `R`, khoảng trống `R + 1`, mỗi khe chứa tối đa `k − 1` (ở đây `k = 3` ⇒ 2).
- Bài nhiều đáp án ⇒ test bằng **validator**.

## 15. Biến thể của bài

1. **Reorganize String** (LeetCode 767): không có 2 chữ liền nhau, dùng hết ký tự ⇒ cùng tham lam heap, điều kiện `L ≤ R + 1`.
2. **Rearrange String k Distance Apart** (LeetCode 358): hai chữ giống nhau cách nhau ≥ k ⇒ heap + hàng đợi chờ.
3. **Task Scheduler** (LeetCode 621): như trên nhưng được chèn "idle".
4. **Tổng quát `m` chữ, tối đa `r` chữ liền** (Bài 5 mục 17): tham lam heap, công thức `min(L + R, r(R + 1) + R)`.
5. **String Without AAA or BBB** (LeetCode 984): bản 2 chữ, luôn dùng hết.
6. **Đếm số chuỗi happy** dài nhất ⇒ DP đếm trên cùng trạng thái của Cách 2.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quay lui | thử mọi chuỗi | hàm mũ | `O(N)` | đối chiếu trên bộ rất nhỏ |
| 2. DP có nhớ | trạng thái `(a,b,c,last,run)` | đa thức nhưng lớn | lớn | kiểm chứng chính xác, đếm số cách |
| 3. Tham lam heap | max còn lại, bị chặn lấy kế | `O(N)` | `O(N)` | **mặc định**, tổng quát cho nhiều loại |
| 4. Tham lam sắp xếp | sắp lại 3 phần tử mỗi bước | `O(N)` | `O(N)` | ít loại, code ngắn |
| 5. Dựng đệ quy | khối `AA` + vách `B` | `O(N)` | `O(N)` | muốn nghĩ theo cấu trúc, chứng minh dễ |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Độ dài kết quả của `longestDiverseString(0, 8, 11)` là bao nhiêu? Còn `(4, 42, 7)`?

### Bài 2 (tính tay)
Chạy tay Cách 3 cho `(a, b, c) = (2, 2, 1)`: viết heap và chuỗi sau mỗi bước.

### Bài 3 (điền TODO)
Hoàn thiện `is_happy_todo(s, a, b, c)`.

### Bài 4 (tự viết lại)
Viết `max_len(a, b, c)` theo công thức **và** viết lại Cách 4 không dùng heap theo cách của bạn; kiểm tra độ dài khớp.

### Bài 5 (mini challenge)
Tổng quát: cho `counts` (dict chữ → số lượng) và `r ≥ 1`. Trả về chuỗi dài nhất dùng tối đa `counts[ch]` chữ `ch`, không có quá `r` chữ giống nhau liền nhau. Viết `longest_general(counts, r)` bằng heap.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def is_happy_todo(s, a, b, c):
    # TODO: chỉ gồm 'a', 'b', 'c'
    # TODO: số lượng mỗi chữ không vượt giới hạn
    # TODO: không có 3 ký tự giống nhau liên tiếp
    pass

# Bài 4 — TODO
def max_len(a, b, c):
    # TODO: L = max, R = tổng còn lại, trả min(L + R, 2 * (R + 1) + R)
    pass

# Bài 5 — TODO
def longest_general(counts, r):
    # TODO: heap (-count, ch); bị chặn khi r ký tự cuối đều là ch
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `(0, 8, 11)`: `L = 11`, `R = 8`. (2) `11 ≤ 2·9 = 18` ⇒ dùng hết ⇒ 19. (3) `(4, 42, 7)`: `L = 42`, `R = 11`, `2·12 = 24 < 42` ⇒ `24 + 11 = 35`.

**Bài 2.** (1) Heap `a2 b2 c1` ⇒ đặt `a`. (2) Rồi `b2 a1 c1` ⇒ đặt `b`; `a1 b1 c1` ⇒ đặt `a` (thứ tự hoà theo chữ cái). (3) Kết quả dùng hết 5 ký tự, ví dụ `"ababc"`.

**Bài 3.** (1) `set(s) <= set("abc")`. (2) `s.count(...)` so với giới hạn. (3) Duyệt `i` từ 0 đến `len − 3`.

**Bài 4.** (1) Công thức như mục 4. (2) Cách 4: sắp xếp, đặt chữ đầu tiên không bị chặn. (3) So với `happy_dp` trên bộ nhỏ.

**Bài 5.** (1) Như Cách 3, thay điều kiện chặn bằng `len(res) >= r and all(x == ch for x in res[-r:])`. (2) Cận trên: `min(L + R, r(R + 1) + R)`. (3) So với DP/brute force trên bộ nhỏ.

## 19. Đáp án tham khảo

In [27]:
# Bài 1, 2
assert len(Solution().longestDiverseString(0, 8, 11)) == 19
assert len(Solution().longestDiverseString(4, 42, 7)) == 35
r = Solution().longestDiverseString(2, 2, 1)
print("Bài 2:", r)
assert is_happy(r, 2, 2, 1) and len(r) == 5
print("Bài 1, 2: đúng")

Bài 2: ababc
Bài 1, 2: đúng


In [28]:
# Bài 3 — đáp án
def is_happy_sol(s, a, b, c):
    if not set(s) <= set("abc"):
        return False
    if s.count("a") > a or s.count("b") > b or s.count("c") > c:
        return False
    for i in range(len(s) - 2):
        if s[i] == s[i + 1] == s[i + 2]:
            return False
    return True

for s, abc, want in [("ccaccbcc", (1, 1, 7), True), ("aaab", (5, 5, 5), False),
                     ("abd", (1, 1, 1), False), ("aab", (1, 1, 0), False), ("", (0, 0, 1), True)]:
    assert is_happy_sol(s, *abc) == want == is_happy(s, *abc)
print("Bài 3: đúng")

Bài 3: đúng


In [29]:
# Bài 4 — đáp án
def max_len_sol(a, b, c):
    L = max(a, b, c)
    R = a + b + c - L
    return min(L + R, 2 * (R + 1) + R)

def happy_sort_sol(a, b, c):
    left = {"a": a, "b": b, "c": c}
    out = []
    while True:
        for ch in sorted(left, key=lambda x: -left[x]):
            if left[ch] and out[-2:] != [ch, ch]:
                out.append(ch); left[ch] -= 1
                break
        else:
            return "".join(out)

for a in range(7):
    for b in range(7):
        for c in range(7):
            if a + b + c:
                r = happy_sort_sol(a, b, c)
                assert is_happy(r, a, b, c) and len(r) == max_len_sol(a, b, c) == len(happy_dp(a, b, c))
print("Bài 4: đúng")

Bài 4: đúng


In [30]:
# Bài 5 — đáp án
def longest_general_sol(counts, r):
    h = [(-n, ch) for ch, n in counts.items() if n > 0]
    heapq.heapify(h)
    res = []
    while h:
        n1, c1 = heapq.heappop(h)
        if len(res) >= r and all(x == c1 for x in res[-r:]):
            if not h:
                break
            n2, c2 = heapq.heappop(h)
            res.append(c2)
            if n2 + 1 < 0:
                heapq.heappush(h, (n2 + 1, c2))
            heapq.heappush(h, (n1, c1))
        else:
            res.append(c1)
            if n1 + 1 < 0:
                heapq.heappush(h, (n1 + 1, c1))
    return "".join(res)

def general_ok(s, counts, r):
    if any(s.count(ch) > n for ch, n in counts.items()) or not set(s) <= set(counts):
        return False
    run = 0
    for i, ch in enumerate(s):
        run = run + 1 if i and s[i - 1] == ch else 1
        if run > r:
            return False
    return True

def general_brute_len(counts, r):
    keys = sorted(counts)
    @lru_cache(maxsize=None)
    def best(state, last, run):
        res = 0
        for i, ch in enumerate(keys):
            if state[i] and not (ch == last and run == r):
                ns = state[:i] + (state[i] - 1,) + state[i + 1:]
                res = max(res, 1 + best(ns, ch, run + 1 if ch == last else 1))
        return res
    return best(tuple(counts[k] for k in keys), "", 0)

rng = random.Random(5)
for _ in range(400):
    m = rng.randint(1, 4)
    counts = {ch: rng.randint(0, 6) for ch in "abcd"[:m]}
    if sum(counts.values()) == 0:
        continue
    r = rng.randint(1, 3)
    s = longest_general_sol(counts, r)
    assert general_ok(s, counts, r) and len(s) == general_brute_len(counts, r), (counts, r, s)
assert longest_general_sol({"a": 1, "b": 1, "c": 7}, 2).count("c") == 6
print(longest_general_sol({"a": 9, "b": 2}, 3), longest_general_sol({"x": 3, "y": 3, "z": 3}, 1))
print("Bài 5: đúng")

aaabaaabaaa xyzxyzxyz
Bài 5: đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tham lam "không quá k−1 chữ liền":**

```python
heap = [(-cnt, ch) for cnt, ch in items if cnt > 0]
heapq.heapify(heap)
res = []
while heap:
    n1, c1 = heapq.heappop(heap)
    if res[-2:] == [c1, c1]:          # bị chặn
        if not heap: break            # hết vách ngăn -> dừng
        n2, c2 = heapq.heappop(heap)
        res.append(c2)
        if n2 + 1 < 0: heapq.heappush(heap, (n2 + 1, c2))
        heapq.heappush(heap, (n1, c1))
    else:
        res.append(c1)
        if n1 + 1 < 0: heapq.heappush(heap, (n1 + 1, c1))
return "".join(res)
```

**Độ dài tối ưu:** `min(L + R, 2(R + 1) + R)`.

**Checklist trước khi submit:**
- [ ] Max-heap bằng số âm.
- [ ] Mỗi lượt đặt **một** chữ, kiểm tra 2 ký tự cuối.
- [ ] Bị chặn + heap rỗng ⇒ `break` (không lặp vô hạn).
- [ ] Trả chữ nhiều nhất về heap sau khi dùng vách ngăn.
- [ ] Bỏ qua chữ có số lượng 0 ngay từ đầu.
- [ ] Test bằng validator + độ dài, không so chuỗi cố định.
- [ ] Đúng chữ ký: `longestDiverseString(self, a: int, b: int, c: int) -> str`.